Clusterização

In [10]:
!pip install pandas
!pip install sentence-transformers
!pip install numpy
!pip install sklearn


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.c

In [11]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [12]:
df = pd.read_csv("data/com_texto_limpo.csv")
embeddings = np.load("data/emb_minilm.npy")

In [13]:
mascara = ~df["multi_claim"]
df_filtrado = df[mascara].reset_index(drop=True)
E = embeddings[mascara]

print(f"Tamanho original: {len(df)} | Após filtro: {len(df_filtrado)}\n")

Tamanho original: 1877 | Após filtro: 1858



In [ ]:
valores_k = [10, 15, 20, 30, 40]
print("--- Teste de Baseline: KMeans ---")
for k in valores_k:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = kmeans.fit_predict(E)
    score = silhouette_score(E, labels, metric="cosine")
    print(f"K={k:2d} | Silhouette Score: {score:.4f}")

--- Teste de Baseline: KMeans ---
K=10 | Silhouette Score: 0.0846
K=15 | Silhouette Score: 0.0851
K=20 | Silhouette Score: 0.0731
K=30 | Silhouette Score: 0.0537
K=40 | Silhouette Score: 0.0561


In [16]:
melhor_k = 15
kmeans_final = KMeans(n_clusters=melhor_k, random_state=42, n_init='auto')
df_filtrado['cluster_kmeans'] = kmeans_final.fit_predict(E)

In [17]:
np.random.seed(42)
clusters_sorteados = np.random.choice(range(melhor_k), size=8, replace=False)

print(f"--- Inspeção Qualitativa (KMeans com K={melhor_k}) ---\n")

--- Inspeção Qualitativa (KMeans com K=15) ---



In [18]:
for c in clusters_sorteados:
    print(f"📌 CLUSTER {c}:")
    
    textos_do_cluster = df_filtrado[df_filtrado['cluster_kmeans'] == c]['Título da checagem']
    
    amostra = textos_do_cluster.sample(min(5, len(textos_do_cluster)), random_state=42)
    
    for texto in amostra:
        print(f"  - {texto}")
    print("-" * 50)

📌 CLUSTER 9:
  - Fotos antigas de canais danificados da transposição do São Francisco circulam fora de contexto
  - MST está se programando para invadir casas de veraneio em Santa Catarina #boato
  - Padre é agredido por bolsonaristas na cidade de Candiota (RS) #boato
  - É #FAKE que TSE envie mensagens de cancelamento de título por irregularidade no CPF
  - É falso que urnas eletrônicas estão sendo preparadas para fraude
--------------------------------------------------
📌 CLUSTER 11:
  - É #FAKE vídeo que mostra Bolsonaro na liderança da pesquisa Ipec divulgada em 15 de agosto de 2022
  - É montagem vídeo que mostra ex-jogador Ronaldo dançando música de apoio a Bolsonaro
  - Imagem de Flávio Bolsonaro com uma camisa contra nordestinos é montagem
  - Bolsonaro exibiu cartaz “Globo lixo” em frente da sede da Globo após debate #boato
  - Vídeo é manipulado para afirmar que Bolsonaro liderou pesquisa Ipec divulgada em 12 de setembro
--------------------------------------------------
📌 CL